In [2]:
import numpy as np
import math as math

In [4]:
def coupon_bond_price(f, coupon_rate, r, t, m = 1):
    c = (f * coupon_rate) / m

    pv_coupons1 = sum(c / ((1 + r / m) ** k) for k in range(1, int(t * m) + 1))
    pv_face1 = f / ((1 + r / m) ** (int(t * m)))
    
    pv_coupons2 = sum(c * math.exp(-r * (k / m)) for k in range(1, int(t * m) + 1))
    pv_face2 = f * math.exp(-r * t)
    
    simple_price = sum(c / (1 + r * (k / m)) for k in range(1, int(t * m) + 1)) + (f / (1 + r * t))

    return {
        "Simple Interest": simple_price,
        "Discrete Compounding": pv_coupons1 + pv_face1,
        "Continuous Compounding": pv_coupons2 + pv_face2
    }

In [5]:
def duration_bonds(market_rate, coupon_rate, fv, t):
    c = coupon_rate * fv
    dur = 0
    durcc = 0
    prices = coupon_bond_price(fv, coupon_rate, market_rate, t, 1)
    for i in range(0, int(t)):
        dur += c * i / (1 + market_rate)**i
        durcc += c * i * math.exp(-market_rate * i)

    return {"compounding":(dur + (c + fv) * t / (1+ market_rate)**t)/prices["Discrete Compounding"],
             "continuous": (durcc + (c + fv)*t*math.exp(-market_rate*t))/prices["Continuous Compounding"]}

In [6]:
duration_bonds(0.05, 0.06, 1000, 5)

{'compounding': 4.477751243087997, 'continuous': 4.476153039318912}

In [3]:
def duration_bond(market_rate, coupon_rate, fv, t, m=1):
    c = (coupon_rate * fv) / m
    n = int(t * m)
    
    # Calculate actual price first
    bond_price_discrete = sum(c / ((1 + market_rate / m) ** k) for k in range(1, n + 1)) + (fv / ((1 + market_rate / m) ** n))
    weighted_pv_discrete = sum((k / m) * c / ((1 + market_rate / m) ** k) for k in range(1, n + 1)) + (t * fv / ((1 + market_rate / m) ** n))
    mac_dur_discrete = weighted_pv_discrete / bond_price_discrete

    # Continuous compounding price & duration
    bond_price_cont = sum(c * math.exp(-market_rate * (k / m)) for k in range(1, n + 1)) + (fv * math.exp(-market_rate * t))
    weighted_pv_cont = sum((k / m) * c * math.exp(-market_rate * (k / m)) for k in range(1, n + 1)) + (t * fv * math.exp(-market_rate * t))
    mac_dur_cont = weighted_pv_cont / bond_price_cont

    return {
        "Bond Price (Discrete)": round(bond_price_discrete, 2),
        "Macaulay Duration (Discrete)": round(mac_dur_discrete, 4),
        "Macaulay Duration (Continuous)": round(mac_dur_cont, 4)
    }

In [4]:
duration_bond(0.05, 0.06, 1000, 5, m=1)

{'Bond Price (Discrete)': 1043.29,
 'Macaulay Duration (Discrete)': 4.4778,
 'Macaulay Duration (Continuous)': 4.4762}

In [1]:
import math

def duration_bbond(market_rate, coupon_rate, fv, t):
    c = coupon_rate * fv
    dur = 0
    durcc = 0
    p = 0
    pcc = 0
    
    # Loop for intermediate coupons (years 1 to t-1)
    for i in range(1, int(t)):
        dur += c * i / (1 + market_rate)**i
        durcc += c * i * math.exp(-market_rate * i)
        p += c / (1 + market_rate)**i
        pcc += c * math.exp(-market_rate * i)

    # Add final coupon + principal at year t to calculate total bond price
    price = p + (c + fv) / (1 + market_rate)**t
    price_cc = pcc + (c + fv) * math.exp(-market_rate * t)

    # Divide by actual bond price instead of fv
    return {"compounding": (dur + (c + fv) * t / (1 + market_rate)**t) / price,
            "continuous": (durcc + (c + fv) * t * math.exp(-market_rate * t)) / price_cc}

In [3]:
duration_bbond(0.05, 0.06, 1000, 5)

{'compounding': 4.477751243087997, 'continuous': 4.476153039318913}